# Phase 17: Advanced Model Validation, Calibration, and Robustness\n\nThis notebook evaluates probability calibration (Brier score, reliability diagrams), threshold sensitivity, customer subgroup performance, and edge-case stress testing for the production churn pipeline.

In [ ]:
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from churn_prediction.config import MODELS_DIR, PROCESSED_DATA_DIR, PHASE_17_REPORTS_DIR, FIGURES_DIR
from churn_prediction.calibration import compute_calibration_curve, evaluate_threshold_sweep
from churn_prediction.robustness import evaluate_subgroup_performance

print('Imports loaded.')

## 1. Load Model & Holdout Evaluation Data

In [ ]:
bundle = joblib.load(MODELS_DIR / 'churn_pipeline_bundle.joblib')
pipeline = bundle['pipeline']
test_df = pd.read_csv(PROCESSED_DATA_DIR / 'test.csv')
X_test = test_df[bundle['feature_names']]
y_test = test_df['Churn Value'].values
y_prob = pipeline.predict_proba(X_test)[:, 1]

print(f'Test samples: {len(X_test)}, Churn prevalence: {y_test.mean():.2%}')

## 2. Probability Calibration Assessment\n\nChecking raw vs calibrated Brier scores and reliability curves.

In [ ]:
prob_true, prob_pred, brier = compute_calibration_curve(y_test, y_prob)
print(f'Uncalibrated HistGradientBoosting Brier Score: {brier:.4f}')

with open(PHASE_17_REPORTS_DIR / 'validation_summary.json') as f:
    val_summary = json.load(f)
print(json.dumps(val_summary['calibration'], indent=2))

## 3. Threshold Tradeoff Analysis

In [ ]:
thresh_df = pd.read_csv(PHASE_17_REPORTS_DIR / 'threshold_analysis.csv')
display(thresh_df[(thresh_df['threshold'] >= 0.20) & (thresh_df['threshold'] <= 0.50)])

## 4. Subgroup Performance Auditing

In [ ]:
subgroup_df = pd.read_csv(PHASE_17_REPORTS_DIR / 'subgroup_performance.csv')
display(subgroup_df.head(10))

## 5. Stress Testing Findings

In [ ]:
extreme_df = pd.read_csv(PHASE_17_REPORTS_DIR / 'numerical_extreme_stress_test.csv')
display(extreme_df)